# Stage 6A.3c — IK reach analysis (oracle targets)

Judges the `stage6a3c_ik_reach.py` runs from their artifacts only (`results/runs/<sweep>/ik_reach_*/steps.csv` + `summary.json`). Nothing here talks to the sim.

**Sections**
- **A. Tracking error** at the hover vs. the tolerance (recomputed from `steps.csv`, cross-checked against `summary.json`)
- **B. Droop vs. lean**: does the error grow with horizontal reach, and which joints carry it?
- **C. Mirror check**: every (x, +y) / (x, −y) pair
- **D. Following error** during the moves: the lag of the actual joints behind the commanded ones, and the time constant it implies
- **E. Actual-path sweep**: the offline floor/cube screens rerun on the path the arm *actually* took (FK of measured joints), plus how far the actual TCP path strayed from the commanded one
- **F. Settling** at the hover: time to within 0.1 mm, and overshoot
- **G. Summary table**, also written to `results/analysis/ik_reach_summary.csv`

**Kernel:** the repo `.venv` (needs `roboticstoolbox` for section E). ROS is not needed.

**Terms used:** *tracking error* = measured TCP minus the commanded target. *Droop* = actual joint angle minus commanded joint angle while holding still (gravity load against the drive's stiffness). *Following error* = the same difference while moving; for a spring–damper drive following a ramp it's roughly speed × a time constant (EE analog: velocity lag of a type-1 loop on a ramp input).

In [ ]:
# ---- Parameters (real values; edit here only) --------------------------------
import os, glob, json, csv, math
RUNS_DIR  = "results/runs"    # relative to this notebook (the repo root); one folder per sweep
RUN_GLOB  = "*/ik_reach_*"   # sweep folder / run folder
EXCLUDE   = []                # run-folder names to leave out, e.g. a known-bad run
OUT_DIR   = "results/analysis"

import sys
# Jupyter kernels (PyCharm's included) don't always put the working directory on the
# module search path the way a script run does. Add it, so the project's own module
# stage6a3c_offline_predict_v2.py (next cell) can be imported.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())
assert os.path.isfile("stage6a3c_offline_predict_v2.py"), \
    "stage6a3c_offline_predict_v2.py not in the working directory -- open this notebook from the repo root"
assert os.path.isdir(RUNS_DIR), f"no {RUNS_DIR}/ here -- open this notebook from the project root"
run_dirs = sorted(d for d in glob.glob(os.path.join(RUNS_DIR, RUN_GLOB))
                  if os.path.isdir(d) and os.path.basename(d) not in EXCLUDE
                  and os.path.isfile(os.path.join(d, "steps.csv"))
                  and os.path.isfile(os.path.join(d, "summary.json")))
print(f"{len(run_dirs)} complete runs:", [os.path.basename(d) for d in run_dirs])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import roboticstoolbox as rtb
import stage6a3c_offline_predict_v2 as op   # same thresholds/frames as the scripts (v1, now in dev_scripts/, has identical ones)
%matplotlib inline
# ^ AFTER the import: the offline module calls matplotlib.use("Agg") (file-only output),
#   which would otherwise silence every plt.show() in this notebook.
print(f"numpy {np.__version__}, roboticstoolbox {rtb.__version__}")
panda = rtb.models.Panda()

CMD = [f"cmd_q{k}" for k in range(1, 8)]
ACT = [f"act_q{k}" for k in range(1, 8)]
SIM_HZ = 60.0

def load_run(d):
    s = json.load(open(os.path.join(d, "summary.json")))
    with open(os.path.join(d, "steps.csv")) as fh:
        rows = list(csv.DictReader(fh))
    cols = {k: np.array([float(r[k]) for r in rows]) for k in rows[0] if k != "phase"}
    cols["phase"] = np.array([r["phase"] for r in rows])
    cx, cy = s["cube_xy"]
    tgt = np.array(s["target_tcp"])
    return dict(name=os.path.basename(d), s=s, c=cols, cx=cx, cy=cy, target=tgt,
                cube_top=float(tgt[2] - op.HOVER_M),
                label=f"({cx:.2f},{cy:+.2f})" + ("c" if "corrected" in s else ""),
                cmd=np.column_stack([cols[k] for k in CMD]),
                act=np.column_stack([cols[k] for k in ACT]),
                tcp=np.column_stack([cols["tcp_x"], cols["tcp_y"], cols["tcp_z"]]))

runs = sorted((load_run(d) for d in run_dirs), key=lambda r: (r["cx"], r["cy"], r["name"]))
print('label suffix "c" = a stage6a3c_ik_reach_corr.py run (sections A-F judge its UNCORRECTED hover; H judges the correction)')
for r in runs:
    ph, n = np.unique(r["c"]["phase"], return_counts=True)
    print(f"{r['label']}  {r['name']:<28} cube top {r['cube_top']:.3f} m   "
          + "  ".join(f"{p} {k}" for p, k in zip(ph, n)))

## A. Tracking error at the hover

In [ ]:
# Copied from stage6a3c_ik_reach.py (not imported: that module needs rclpy).
def quat_xyzw_to_R(x, y, z, w):
    n = math.sqrt(x*x + y*y + z*z + w*w); x, y, z, w = x/n, y/n, z/n, w/n
    return np.array([[1-2*(y*y+z*z), 2*(x*y-z*w),   2*(x*z+y*w)],
                     [2*(x*y+z*w),   1-2*(x*x+z*z), 2*(y*z-x*w)],
                     [2*(x*z-y*w),   2*(y*z+x*w),   1-2*(x*x+y*y)]])

def mean_rotation(Rs):
    U, _, Vt = np.linalg.svd(np.mean(Rs, axis=0)); R = U @ Vt
    if np.linalg.det(R) < 0: U[:, -1] *= -1; R = U @ Vt
    return R

def hold_stats(r, phase, n=30):
    idx = np.where(r["c"]["phase"] == phase)[0][-n:]
    c = r["c"]
    Rs = [quat_xyzw_to_R(c["hand_qx"][i], c["hand_qy"][i], c["hand_qz"][i], c["hand_qw"][i]) for i in idx]
    return dict(tcp=r["tcp"][idx].mean(0), R=mean_rotation(np.array(Rs)),
                droop=r["act"][idx].mean(0) - r["cmd"][idx[-1]])

TOL = runs[0]["s"]["config"]
TOL_MM, TOL_TILT, TOL_HEAD = TOL["tol_pos_mm"], TOL["tol_tilt_deg"], TOL["tol_heading_deg"]
print(f"tolerance: {TOL_MM:g} mm / {TOL_TILT:g} deg tilt / {TOL_HEAD:g} deg heading\n")
# SOURCE OF THE HOVER NUMBERS. summary.json holds them at full precision (computed by
# the script from the live messages). steps.csv is written at 4 decimal places in
# METRES and RADIANS, i.e. 0.1 mm and 0.1 mrad steps -- too coarse for mm/mrad-level
# judgments. So: use summary.json when it has the "hover" block; fall back to
# recomputing from steps.csv (and say so) only when it doesn't.
print(f"{'cube':<14}{'err mm':>8}{'dx':>8}{'dy':>8}{'dz':>8}{'tilt':>8}{'head':>8}"
      f"{'% tol':>7}  PASS  source (csv recompute differs by)")
for r in runs:
    h = hold_stats(r, "hold_hover")
    e_csv = h["tcp"] - r["target"]
    hv = r["s"].get("hover")
    if hv:
        e = np.array(hv["err_xyz_m"]); r["tilt"], r["head"] = hv["tilt_deg"], hv["heading_err_deg"]
        r["droop"] = np.array(hv["droop_rad"])
        src = f"summary.json ({1000*np.linalg.norm(e_csv - e):.3f} mm)"
    else:
        e = e_csv; r["tilt"] = op.tilt_deg(h["R"])
        r["head"] = abs(op.wrap180(op.heading_deg(h["R"]) - r["s"]["heading_deg"]))
        r["droop"] = h["droop"]; src = "steps.csv (0.1 mm / 0.1 mrad resolution)"
    r["err"], r["err_mm"] = e, 1000 * float(np.linalg.norm(e))
    r["pass"] = r["err_mm"] <= TOL_MM and r["tilt"] <= TOL_TILT and r["head"] <= TOL_HEAD
    print(f"{r['label']:<14}{r['err_mm']:>8.4f}{1000*e[0]:>8.3f}{1000*e[1]:>8.3f}{1000*e[2]:>8.3f}"
          f"{r['tilt']:>8.4f}{r['head']:>8.4f}{100*r['err_mm']/TOL_MM:>6.0f}%  "
          f"{'PASS' if r['pass'] else 'FAIL'}  {src}")

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(runs))
for k, (lab, col) in enumerate((("dx", "tab:blue"), ("dy", "tab:orange"), ("dz", "tab:green"))):
    ax.bar(x + (k - 1) * 0.22, [1000 * r["err"][k] for r in runs], 0.22, label=lab, color=col)
ax.plot(x, [r["err_mm"] for r in runs], "kD", label="|error|")
ax.axhline(TOL_MM, color="tab:red", ls=":", label=f"tolerance {TOL_MM:g} mm")
ax.axhline(0, color="0.5", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([r["label"] for r in runs])
ax.set_ylabel("mm"); ax.set_title("A. TCP tracking error at the hover (measured − target)")
ax.legend(fontsize=8, ncol=5); ax.grid(True, axis="y", alpha=0.3); plt.show()

## B. Droop vs. lean
Horizontal reach r = √(x² + y²) of the cube. If droop is gravity against a stiff spring, the error should grow with how far the arm leans out.

In [ ]:
for r in runs:
    r["reach"] = math.hypot(r["cx"], r["cy"])
order = sorted(runs, key=lambda r: r["reach"])
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
ax = axs[0]
ax.plot([r["reach"] for r in order], [r["err_mm"] for r in order], "kD-", label="|error|")
ax.plot([r["reach"] for r in order], [-1000 * r["err"][2] for r in order], "o--", color="tab:green", label="−dz")
for r in order:
    ax.annotate(r["label"], (r["reach"], r["err_mm"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.axhline(TOL_MM, color="tab:red", ls=":")
ax.set_xlabel("horizontal reach of the cube (m)"); ax.set_ylabel("mm")
ax.set_title("TCP error vs. reach"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax = axs[1]
for j in range(7):
    ax.plot([r["reach"] for r in order], [1000 * r["droop"][j] for r in order], "o-", label=f"j{j+1}")
ax.set_xlabel("horizontal reach of the cube (m)"); ax.set_ylabel("actual − commanded (mrad)")
ax.set_title("Joint droop at the hover"); ax.legend(fontsize=8, ncol=4); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("droop at the hover (mrad):")
for r in order:
    print(f"  {r['label']}  r {r['reach']:.4f} m  " + "  ".join(f"j{k+1} {1000*d:+7.3f}" for k, d in enumerate(r["droop"])))

## C. Mirror check
For each (x, +y)/(x, −y) pair: dx and dz should match, dy should flip; joints 1/3/5/7 droop should flip sign, joints 2/4/6 should match (the mirror image of the arm).

In [ ]:
pairs = [(a, b) for a in runs for b in runs if a["cy"] > 0 and abs(a["cx"] - b["cx"]) < 1e-9
         and abs(a["cy"] + b["cy"]) < 1e-9 and a["label"][-1:] == b["label"][-1:]]
if not pairs:
    print("no mirror pairs in this run set")
for a, b in pairs:
    flip = np.array([1, -1, 1])
    e_mis = 1000 * float(np.max(np.abs(a["err"] - flip * b["err"])))
    jflip = np.array([-1, 1, -1, 1, -1, 1, -1])
    d_mis = 1000 * float(np.max(np.abs(a["droop"] - jflip * b["droop"])))
    print(f"{a['label']} vs {b['label']}: TCP error mismatch (after mirroring) {e_mis:.4f} mm, "
          f"joint droop mismatch {d_mis:.4f} mrad  -> {'MIRROR OK' if e_mis < 0.05 and d_mis < 0.05 else 'CHECK'}")

## D. Following error during the moves

In [ ]:
fig, axs = plt.subplots(len(runs), 1, figsize=(12, 2.6 * len(runs)), sharex=True)
axs = np.atleast_1d(axs)
print(f"{'cube':<14}{'move':<12}{'peak lag rad':>13}{'joint':>6}{'peak speed':>11}{'lag/speed s':>12}")
for ax, r in zip(axs, runs):
    t = np.arange(len(r["cmd"])) / SIM_HZ
    lag = r["act"] - r["cmd"]
    for j in range(7):
        ax.plot(t, lag[:, j], lw=1.2, label=f"j{j+1}")
    for ph in ("move_ready", "move_hover"):
        idx = np.where(r["c"]["phase"] == ph)[0]
        ax.axvspan(t[idx[0]], t[idx[-1]], color="0.92")
        v = np.vstack([np.zeros(7), np.diff(r["cmd"][idx], axis=0) * SIM_HZ])
        i, j = np.unravel_index(np.abs(lag[idx]).argmax(), lag[idx].shape)
        tau = abs(lag[idx][i, j]) / max(abs(v[i, j]), 1e-9)
        print(f"{r['label']:<14}{ph:<12}{abs(lag[idx][i, j]):>13.4f}{'j'+str(j+1):>6}"
              f"{abs(v[i, j]):>11.4f}{tau:>12.4f}")
    ax.set_ylabel("act − cmd (rad)"); ax.set_title(f"{r['label']}  (grey = moving)", fontsize=9)
    ax.grid(True, alpha=0.3)
axs[0].legend(fontsize=7, ncol=7, loc="lower right"); axs[-1].set_xlabel("sim time (s)")
plt.tight_layout(); plt.show()

## E. Actual-path sweep
The offline screens (`op.FLOOR_CLEAR_MIN`, `op.TCP_FLOOR_CLEAR_MIN`, `op.CUBE_CLEAR_MIN`, the same named frames) rerun on FK of the **measured** joints, plus the largest gap between the actual and the commanded TCP path. Takes a little while (one FK call per frame per step).

In [ ]:
def cube_clear(p, cx, cy, size):
    h = size / 2
    lo, hi = np.array([cx - h, cy - h, 0.0]), np.array([cx + h, cy + h, size])
    return float(np.linalg.norm(np.maximum(np.maximum(lo - p, p - hi), 0.0)))

print(f"{'cube':<14}{'min link z':>11}{'(frame)':>20}{'TCP min z':>10}{'TCP-cube':>9}"
      f"{'path gap mm':>12}  screens")
for r in runs:
    moving = np.isin(r["c"]["phase"], ["move_ready", "hold_ready", "move_hover", "hold_hover"])
    lz, lname, gap = np.inf, "", 0.0
    for q_a, q_c in zip(r["act"][moving], r["cmd"][moving]):
        for f in op.SWEEP_FRAMES:
            z = panda.fkine(q_a, end=f).t[2]
            if z < lz: lz, lname = float(z), f
        gap = max(gap, float(np.linalg.norm(panda.fkine(q_a).t - panda.fkine(q_c).t)))
    P = r["tcp"][moving]
    tz = float(P[:, 2].min())
    cc = min(cube_clear(p, r["cx"], r["cy"], r["cube_top"]) for p in P)
    ok = lz >= op.FLOOR_CLEAR_MIN and tz >= op.TCP_FLOOR_CLEAR_MIN and cc >= op.CUBE_CLEAR_MIN
    r["sweep"] = dict(link_z=lz, frame=lname, tcp_z=tz, cube=cc, gap_mm=1000 * gap, ok=ok)
    print(f"{r['label']:<14}{lz:>11.4f}{lname:>20}{tz:>10.4f}{cc:>9.4f}{1000*gap:>12.2f}  "
          f"{'ok' if ok else 'FAIL'}")
print(f"\nscreens: link z >= {op.FLOOR_CLEAR_MIN} m, TCP z >= {op.TCP_FLOOR_CLEAR_MIN} m, "
      f"TCP-cube >= {op.CUBE_CLEAR_MIN} m")

## F. Settling at the hover

In [ ]:
print(f"{'cube':<14}{'to 1 mm s':>10}{'to 0.1 mm s':>12}{'overshoot below final z mm':>28}")
for r in runs:
    idx = np.where(r["c"]["phase"] == "hold_hover")[0]
    P = r["tcp"][idx]; fin = P[-30:].mean(0)
    d = 1000 * np.linalg.norm(P - fin, axis=1)
    def t_to(th):
        k = np.where(d > th)[0]
        return 0.0 if len(k) == 0 else (k[-1] + 1) / SIM_HZ
    r["settle_01"] = t_to(0.1)
    r["overshoot_mm"] = 1000 * max(0.0, fin[2] - P[:, 2].min())
    print(f"{r['label']:<14}{t_to(1.0):>10.2f}{r['settle_01']:>12.2f}{r['overshoot_mm']:>28.4f}")

## H. Droop correction (`stage6a3c_ik_reach_corr.py` runs only)
Uncorrected → corrected tracking error per run and per iteration. **Prediction (made before these runs):** one iteration leaves well under 0.2 mm. If joint1 has friction, the y = ±0.30 runs may keep up to ~0.5 mm in **y**.

In [ ]:
cruns = [r for r in runs if "corrected" in r["s"]]
if not cruns:
    print("no correction runs yet")
for r in cruns:
    it = r["s"]["corrected"]["iterations"]
    line = f"{r['label']:<15} uncorrected {r['err_mm']:.4f} mm"
    for c in it:
        e = 1000 * np.array(c["err_xyz_m"])
        line += (f"  -> iter {c['iteration']}: {c['err_mm']:.4f} mm "
                 f"(dx {e[0]:+.4f}, dy {e[1]:+.4f}, dz {e[2]:+.4f})")
    print(line + f"   {'PASS' if r['s']['corrected']['PASS'] else 'FAIL'}")
    c = it[-1]
    print(" " * 15 + " droop change after correction (mrad): "
          + "  ".join(f"j{k+1} {1000*(c['droop_rad'][k] - r['droop'][k]):+.4f}" for k in range(7)))
if cruns:
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(cruns))
    ax.bar(x - 0.2, [r["err_mm"] for r in cruns], 0.4, label="uncorrected", color="0.6")
    ax.bar(x + 0.2, [r["s"]["corrected"]["err_mm"] for r in cruns], 0.4, label="corrected", color="tab:blue")
    ax.axhline(TOL_MM, color="tab:red", ls=":", label=f"tolerance {TOL_MM:g} mm")
    ax.set_yscale("log"); ax.set_ylabel("tracking error (mm, log)")
    ax.set_xticks(x); ax.set_xticklabels([r["label"] for r in cruns])
    ax.set_title("H. Droop correction: tracking error before and after")
    ax.legend(fontsize=8); ax.grid(True, axis="y", which="both", alpha=0.3); plt.show()

## G. Summary

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
hdr = ["run", "cube_x", "cube_y", "cube_top_m", "reach_m", "err_mm", "dx_mm", "dy_mm", "dz_mm",
       "tilt_deg", "heading_err_deg", "pct_of_tol", "PASS"] + [f"droop_j{k}_mrad" for k in range(1, 8)] \
      + ["sweep_min_link_z_m", "sweep_tcp_min_z_m", "sweep_tcp_cube_m", "path_gap_mm", "sweep_ok",
         "settle_0p1mm_s", "overshoot_mm", "corr_iterations", "corrected_err_mm", "corrected_PASS"]
rows = []
for r in runs:
    sw = r["sweep"]
    rows.append([r["name"], r["cx"], r["cy"], r["cube_top"], r["reach"], r["err_mm"],
                 *(1000 * r["err"]), r["tilt"], r["head"], 100 * r["err_mm"] / TOL_MM, r["pass"],
                 *(1000 * r["droop"]), sw["link_z"], sw["tcp_z"], sw["cube"], sw["gap_mm"], sw["ok"],
                 r["settle_01"], r["overshoot_mm"],
                 len(r["s"]["corrected"]["iterations"]) if "corrected" in r["s"] else 0,
                 r["s"]["corrected"]["err_mm"] if "corrected" in r["s"] else float("nan"),
                 r["s"]["corrected"]["PASS"] if "corrected" in r["s"] else ""])
path = os.path.join(OUT_DIR, "ik_reach_summary.csv")
op.write_csv(path, hdr, rows)
n_pass = sum(r["pass"] for r in runs); n_sw = sum(r["sweep"]["ok"] for r in runs)
worst = max(runs, key=lambda r: r["err_mm"])
print(f"tracking PASS {n_pass}/{len(runs)}   actual-path screens ok {n_sw}/{len(runs)}")
print(f"worst error {worst['err_mm']:.4f} mm at {worst['label']} = "
      f"{100*worst['err_mm']/TOL_MM:.0f}% of the {TOL_MM:g} mm tolerance")
print(f"wrote {path}")